# ESM2 Subcellular Localization Prediction (Clean Pipeline)

This notebook builds a clean, reproducible protein-level classifier to predict subcellular localization.

## What this notebook does
- Loads UniProt table and extracts definitive single-compartment labels (9 classes).
- Loads precomputed ESM2 embeddings (1280-dim, max pool and attention pool).
- Trains and compares Logistic Regression and MLP models (both pooling strategies).
- Evaluates accuracy, macro F1, weighted F1, and per-class precision / recall / F1.
- Saves clean plots and summary tables to `results_subcellular_ESM2/`.

In [ ]:
from pathlib import Path
import re
import random
import base64

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import umap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
    ConfusionMatrixDisplay,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
PROJECT_DIR = Path.cwd()

candidate_data_dirs = [
    (PROJECT_DIR / ".." / "all_uniref").resolve(),
    (PROJECT_DIR / ".." / ".." / "all_uniref").resolve(),
    Path("/home/saishyam/Protein_dynamics/all_uniref"),
    Path("/nfs/turbo/umms-mcieslik/saishyam/Protein_dynamics/all_uniref"),
]
DATA_DIR    = next((p for p in candidate_data_dirs if p.exists()), candidate_data_dirs[0])
RESULTS_DIR = PROJECT_DIR / "results_subcellular_ESM2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

UNIPROT_FILE = DATA_DIR / "uniprotkb_AND_reviewed_true_AND_model_o_2026_01_16.tsv"
EMBED_FILE   = DATA_DIR / "esm2_embedding_cache" / "pooled_embeddings.npz"

print(f"Project directory  : {PROJECT_DIR}")
print(f"Data directory     : {DATA_DIR}")
print(f"Results directory  : {RESULTS_DIR}")
print(f"UniProt file exists: {UNIPROT_FILE.exists()}")
print(f"Embedding file exists: {EMBED_FILE.exists()}")

In [ ]:
all_uniref = pd.read_csv(UNIPROT_FILE, sep="\t")
print(f"UniProt table shape: {all_uniref.shape}")
all_uniref.head()

In [ ]:
# ── Canonical compartments (regex-based) ────────────────────────────────────
LOCATION_REGEX = {
    "nucleus":               r"\bnucleus\b",
    "cytoplasm":             r"\bcytoplasm\b",
    "mitochondrion":         r"\bmitochond",
    "endoplasmic_reticulum": r"\bendoplasmic reticulum\b",
    "golgi":                 r"\bgolgi\b",
    "cell_membrane":         r"\bcell membrane\b|\bplasma membrane\b",
    "extracellular":         r"\bextracellular\b|\bsecreted\b",
    "lysosome":              r"\blysosome\b",
    "peroxisome":            r"\bperoxisome\b",
}

# Exclude ambiguous evidence terms
AMBIGUOUS_TERMS = [
    "by similarity", "probable", "potential",
    "may be", "likely", "according to", "predicted",
]


def extract_single_location(text):
    """Return a single compartment label if exactly one compartment is found
    and no ambiguous evidence qualifiers are present; otherwise return None."""
    if pd.isna(text):
        return None
    text_l = text.lower()
    for term in AMBIGUOUS_TERMS:
        if term in text_l:
            return None
    matched = {label for label, pattern in LOCATION_REGEX.items()
               if re.search(pattern, text_l)}
    return matched.pop() if len(matched) == 1 else None


all_uniref["subcellular_label"] = (
    all_uniref["Subcellular location [CC]"].apply(extract_single_location)
)

localization_df = (
    all_uniref[all_uniref["subcellular_label"].notna()]
    .loc[:, ["Entry", "Protein names", "Sequence", "subcellular_label"]]
    .rename(columns={"Entry": "protein_id", "Protein names": "protein_name"})
    .drop_duplicates(subset="protein_name", keep="first")
    .reset_index(drop=True)
)

assert localization_df["protein_id"].is_unique
assert localization_df["subcellular_label"].notna().all()

vc = localization_df["subcellular_label"].value_counts()
class_counts_df = pd.DataFrame({"Class": vc.index, "Count": vc.values})
class_counts_df["Fraction"] = class_counts_df["Count"] / class_counts_df["Count"].sum()

print(f"Total proteins with definitive single-compartment label: {len(localization_df)}")
display(class_counts_df)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    data=class_counts_df, x="Class", y="Count",
    hue="Class", legend=False, palette="tab10", ax=ax,
)
ax.set_title("Subcellular Location Class Distribution")
ax.set_xlabel("")
ax.set_ylabel("Number of proteins")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
for i, row in class_counts_df.iterrows():
    ax.text(i, row["Count"] * 1.01,
            f"{int(row['Count'])}\n({row['Fraction']:.1%})",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
class_dist_path = RESULTS_DIR / "class_distribution.png"
plt.savefig(class_dist_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {class_dist_path}")

In [ ]:
cache     = np.load(EMBED_FILE, allow_pickle=True)
max_pool  = cache["max"].item()
attn_pool = cache["attn"].item()


def collect_embeddings(df, max_pool, attn_pool):
    """Collect aligned (X_max, X_attn, y) arrays for all proteins
    that have entries in both pooling dicts."""
    X_max_list, X_attn_list, y_list = [], [], []
    missing = 0
    for pid, label in zip(df["protein_id"], df["subcellular_label"]):
        v_max  = max_pool.get(pid)
        v_attn = attn_pool.get(pid)
        if v_max is not None and v_attn is not None:
            X_max_list.append(v_max)
            X_attn_list.append(v_attn)
            y_list.append(label)
        else:
            missing += 1
    if missing:
        print(f"Missing embeddings: {missing} proteins skipped")
    return (
        np.stack(X_max_list).astype(np.float32),
        np.stack(X_attn_list).astype(np.float32),
        np.array(y_list),
    )


X_max, X_attn, y_str = collect_embeddings(localization_df, max_pool, attn_pool)
assert X_max.shape[1]  == 1280, f"Expected 1280-dim, got {X_max.shape[1]}"
assert X_attn.shape[1] == 1280

print(f"Max-pool  embeddings : {X_max.shape}")
print(f"Attn-pool embeddings : {X_attn.shape}")
print(f"Labels               : {y_str.shape}")

In [ ]:
le = LabelEncoder()
y_enc       = le.fit_transform(y_str)
class_names = le.classes_
n_classes   = len(class_names)

print(f"Classes ({n_classes}): {class_names}")


def make_split(X, y_enc):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_enc, test_size=0.2, stratify=y_enc, random_state=SEED
    )
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr).astype(np.float32)
    X_te   = scaler.transform(X_te).astype(np.float32)
    return X_tr, X_te, y_tr, y_te


split_data = {
    "max":  make_split(X_max,  y_enc),
    "attn": make_split(X_attn, y_enc),
}


def evaluate_multiclass(y_true, y_pred):
    acc     = accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_wgt  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    prec, rec, f1_pc, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    return {
        "accuracy":           acc,
        "macro_f1":           f1_mac,
        "weighted_f1":        f1_wgt,
        "per_class_precision": prec,
        "per_class_recall":   rec,
        "per_class_f1":       f1_pc,
        "support":            support,
    }

In [ ]:
results    = []
pred_store = {}

for pooling_name, (Xtr, Xte, ytr, yte) in split_data.items():
    clf = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=SEED,
    )
    clf.fit(Xtr, ytr)
    y_pred = clf.predict(Xte)

    m = evaluate_multiclass(yte, y_pred)
    results.append({
        "model":        "logistic_regression",
        "pooling":      pooling_name,
        "accuracy":     m["accuracy"],
        "macro_f1":     m["macro_f1"],
        "weighted_f1":  m["weighted_f1"],
        "n_test":       len(yte),
    })
    pred_store[("logistic_regression", pooling_name)] = {
        "y_true": yte, "y_pred": y_pred, "metrics": m,
    }
    print(f"LR | {pooling_name:4s}  acc={m['accuracy']:.4f}  "
          f"macro_f1={m['macro_f1']:.4f}  weighted_f1={m['weighted_f1']:.4f}")

print("Logistic regression training complete.")

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, n_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(Xtr, ytr, Xte, yte, n_classes, epochs=30, batch_size=64, lr=1e-3):
    Xtr_t = torch.from_numpy(Xtr)
    ytr_t = torch.from_numpy(ytr)
    Xte_t = torch.from_numpy(Xte)

    train_loader = DataLoader(
        TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True
    )

    counts  = np.bincount(ytr, minlength=n_classes)
    weights = torch.tensor(
        counts.sum() / np.maximum(counts, 1), dtype=torch.float32
    ).to(device)

    model     = MLP(Xtr.shape[1], n_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    model.train()
    for _ in range(epochs):
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
        losses.append(running / len(train_loader))

    model.eval()
    with torch.no_grad():
        logits = model(Xte_t.to(device))
        y_pred = torch.argmax(logits, dim=1).cpu().numpy()

    return y_pred, losses


training_loss = {}

for pooling_name, (Xtr, Xte, ytr, yte) in split_data.items():
    y_pred, losses = train_mlp(Xtr, ytr, Xte, yte, n_classes)

    m = evaluate_multiclass(yte, y_pred)
    results.append({
        "model":        "mlp",
        "pooling":      pooling_name,
        "accuracy":     m["accuracy"],
        "macro_f1":     m["macro_f1"],
        "weighted_f1":  m["weighted_f1"],
        "n_test":       len(yte),
    })
    pred_store[("mlp", pooling_name)] = {
        "y_true": yte, "y_pred": y_pred, "metrics": m,
    }
    training_loss[pooling_name] = losses
    print(f"MLP | {pooling_name:4s}  acc={m['accuracy']:.4f}  "
          f"macro_f1={m['macro_f1']:.4f}  weighted_f1={m['weighted_f1']:.4f}")

print("MLP training complete.")

In [ ]:
results_df = pd.DataFrame(results)[
    ["model", "pooling", "accuracy", "macro_f1", "weighted_f1", "n_test"]
]
results_df = results_df.sort_values(
    ["macro_f1", "accuracy"], ascending=False
).reset_index(drop=True)

csv_path = RESULTS_DIR / "summary_metrics.csv"
md_path  = RESULTS_DIR / "summary_metrics.md"
results_df.to_csv(csv_path, index=False)

header    = "| " + " | ".join(results_df.columns) + " |"
separator = "|" + "|".join(["---"] * len(results_df.columns)) + "|"
fmt_cols  = {"accuracy", "macro_f1", "weighted_f1"}

def _fmt(col, v):
    return f"{v:.4f}" if col in fmt_cols else str(v)

rows_md = [
    "| " + " | ".join(_fmt(c, v) for c, v in zip(results_df.columns, row)) + " |"
    for row in results_df.to_numpy()
]
md_path.write_text("\n".join([header, separator] + rows_md))

display(results_df.style.format({
    "accuracy":    "{:.4f}",
    "macro_f1":    "{:.4f}",
    "weighted_f1": "{:.4f}",
}))
print(f"Saved: {csv_path}")
print(f"Saved: {md_path}")

In [ ]:
cmap_tab = matplotlib.colormaps.get_cmap("tab10")

# ── 1. Confusion matrices (2×2, normalised) ──────────────────────────────────
keys_order = [
    ("logistic_regression", "max"),
    ("logistic_regression", "attn"),
    ("mlp", "max"),
    ("mlp", "attn"),
]

fig, axes = plt.subplots(2, 2, figsize=(20, 16))
for ax, key in zip(axes.ravel(), keys_order):
    model_name, pooling = key
    d  = pred_store[key]
    cm = confusion_matrix(d["y_true"], d["y_pred"], normalize="true")
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format=".2f", colorbar=False)
    ax.set_title(f"{model_name} | {pooling}", fontsize=13)
plt.suptitle("Normalised Confusion Matrices — Subcellular Localisation", fontsize=16)
plt.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrices.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

# ── 2. Per-class metrics — best model (MLP + attn) ───────────────────────────
best_key = ("mlp", "attn")
m        = pred_store[best_key]["metrics"]
x        = np.arange(n_classes)
width    = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width, m["per_class_precision"], width, label="Precision", color="#1f77b4")
ax.bar(x,         m["per_class_recall"],    width, label="Recall",    color="#ff7f0e")
ax.bar(x + width, m["per_class_f1"],        width, label="F1-score",  color="#2ca02c")
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Per-class Precision / Recall / F1 — MLP (attention pooling)")
ax.legend()
plt.tight_layout()
pc_path = RESULTS_DIR / "per_class_metrics.png"
plt.savefig(pc_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {pc_path}")

# ── 3. MLP training loss ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(training_loss["max"],  label="MLP | max",  linewidth=2)
ax.plot(training_loss["attn"], label="MLP | attn", linewidth=2)
ax.set_title("MLP Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("CrossEntropyLoss")
ax.legend()
plt.tight_layout()
loss_path = RESULTS_DIR / "mlp_training_loss.png"
plt.savefig(loss_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {loss_path}")

In [ ]:
# Subsample for speed if the dataset is large
MAX_VIS = 5000
if len(X_attn) > MAX_VIS:
    rng     = np.random.default_rng(SEED)
    idx_vis = rng.choice(len(X_attn), size=MAX_VIS, replace=False)
    X_vis, y_vis = X_attn[idx_vis], y_enc[idx_vis]
    print(f"Subsampling {MAX_VIS} of {len(X_attn)} proteins for projections")
else:
    X_vis, y_vis = X_attn, y_enc

pca_2d  = PCA(n_components=2, random_state=SEED).fit_transform(X_vis)
tsne_2d = TSNE(n_components=2, init="pca", learning_rate="auto",
               perplexity=30, random_state=SEED).fit_transform(X_vis)
umap_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                    random_state=SEED).fit_transform(X_vis)

cmap_colors = matplotlib.colormaps.get_cmap("tab10")

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (title, emb) in zip(axes, [
    ("PCA (Attention embeddings)",   pca_2d),
    ("t-SNE (Attention embeddings)", tsne_2d),
    ("UMAP (Attention embeddings)",  umap_2d),
]):
    for cls_idx, cls_name in enumerate(class_names):
        mask = y_vis == cls_idx
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   s=14, alpha=0.65, color=cmap_colors(cls_idx),
                   label=cls_name, zorder=2)
    ax.set_title(title)
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.02, 0.5),
           title="Subcellular\nlocation", fontsize=9)
plt.suptitle("Dimensionality Reduction of Protein Embeddings\nSubcellular Localisation",
             fontsize=14)
plt.tight_layout(rect=[0, 0, 0.88, 1])
emb_path = RESULTS_DIR / "embedding_projection_pca_tsne_umap.png"
plt.savefig(emb_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {emb_path}")

In [ ]:
# ── Analysis summary ──────────────────────────────────────────────────────────
best_row = results_df.iloc[0]

summary_lines = [
    "ANALYSIS SUMMARY",
    "================",
    f"Model : ESM2 (1280-dim embeddings)",
    f"Task  : Multi-class subcellular localisation ({n_classes} classes)",
    "",
    f"Best model by macro F1: {best_row['model']} + {best_row['pooling']}",
    f"Accuracy    : {best_row['accuracy']:.4f}",
    f"Macro F1    : {best_row['macro_f1']:.4f}",
    f"Weighted F1 : {best_row['weighted_f1']:.4f}",
    "",
    "Dataset:",
    f"  Total proteins with definitive labels: {len(localization_df)}",
    f"  Number of classes: {n_classes}",
    f"  Classes: {', '.join(class_names)}",
    "",
    "Interpretation:",
    "- Macro F1 is emphasised because classes are imbalanced.",
    "- Weighted F1 favours performance on majority classes.",
    "- Confusion matrices (normalised) reveal which compartments are confused.",
    "- Per-class metrics (MLP + attn) identify hard-to-classify compartments.",
]

analysis_path = RESULTS_DIR / "analysis_summary.txt"
analysis_path.write_text("\n".join(summary_lines))
print("\n".join(summary_lines))
print(f"\nSaved: {analysis_path}")

# ── HTML interpretation report ─────────────────────────────────────────────────
def img_b64(path):
    with open(path, "rb") as fh:
        return base64.b64encode(fh.read()).decode()

imgs = {
    "class_dist":  RESULTS_DIR / "class_distribution.png",
    "conf_mat":    RESULTS_DIR / "confusion_matrices.png",
    "per_class":   RESULTS_DIR / "per_class_metrics.png",
    "train_loss":  RESULTS_DIR / "mlp_training_loss.png",
    "projections": RESULTS_DIR / "embedding_projection_pca_tsne_umap.png",
}

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>ESM2 Subcellular Localisation — Interpretation Report</title>
  <style>
    body  { font-family: Arial, sans-serif; max-width: 1100px; margin: auto;
             padding: 24px; background: #fafafa; }
    h1    { color: #2c3e50; border-bottom: 2px solid #2c3e50; padding-bottom: 8px; }
    h2    { color: #34495e; margin-top: 36px; }
    img   { max-width: 100%; border-radius: 6px;
             box-shadow: 0 2px 8px #ccc; margin: 12px 0; }
    table { border-collapse: collapse; width: 100%; margin: 12px 0; }
    th, td { border: 1px solid #ddd; padding: 8px 12px; text-align: left; }
    th    { background: #2c3e50; color: white; }
    tr:nth-child(even) { background: #f2f2f2; }
    .note { background: #eaf4fb; border-left: 4px solid #2980b9;
             padding: 10px 16px; margin: 12px 0; border-radius: 4px; }
  </style>
</head>
<body>
<h1>ESM2 Subcellular Localisation — Interpretation Report</h1>

<h2>1. Task Overview</h2>
<p>Multi-class subcellular localisation prediction using <strong>ESM2 protein language
model embeddings</strong> (1280-dim). The goal is to predict one of {n_classes}
subcellular compartments for each protein, using logistic regression and MLP classifiers
trained on precomputed embeddings (max-pool and attention-pool variants).</p>

<h2>2. Class Distribution</h2>
<img src="data:image/png;base64,{img_b64(imgs['class_dist'])}" alt="Class Distribution">
<p>The dataset is imbalanced across compartments; <em>nucleus</em> and <em>cytoplasm</em>
are typically the most populated classes.  Class-weighted loss and macro F1 are used
to give fair weight to minority compartments.</p>

<h2>3. Model Performance Summary</h2>
{results_df.to_html(index=False, float_format=lambda x: f'{x:.4f}')}
<div class="note">
  <strong>Best model:</strong> {best_row['model']} + {best_row['pooling']} —
  Accuracy {best_row['accuracy']:.4f},
  Macro&nbsp;F1 {best_row['macro_f1']:.4f},
  Weighted&nbsp;F1 {best_row['weighted_f1']:.4f}
</div>

<h2>4. Normalised Confusion Matrices</h2>
<img src="data:image/png;base64,{img_b64(imgs['conf_mat'])}" alt="Confusion Matrices">
<p>Diagonal entries show the per-class recall (fraction of proteins correctly classified
for each compartment). Off-diagonal entries reveal systematic confusions between
structurally or functionally similar compartments (e.g. nucleus ↔ cytoplasm,
ER ↔ Golgi).</p>

<h2>5. Per-class Precision / Recall / F1 (MLP + attention pooling)</h2>
<img src="data:image/png;base64,{img_b64(imgs['per_class'])}" alt="Per-class Metrics">
<p>Compartments with low recall are harder to distinguish — often because they share
functional sequence motifs with other compartments or are under-represented in the
training set.</p>

<h2>6. MLP Training Loss</h2>
<img src="data:image/png;base64,{img_b64(imgs['train_loss'])}" alt="Training Loss">
<p>Smooth, monotonically decreasing loss curves indicate stable training.
Both pooling strategies converge within 30 epochs.</p>

<h2>7. Embedding Projections (PCA · t-SNE · UMAP)</h2>
<img src="data:image/png;base64,{img_b64(imgs['projections'])}" alt="Embedding Projections">
<p>Dimensionality projections of attention-pooled ESM2 embeddings.
Visible clustering by subcellular compartment demonstrates that the embeddings encode
biologically meaningful structural and functional information relevant to
protein localisation.</p>

<hr>
<p style="color:#888; font-size:12px;">
  Generated automatically by the ESM2 Subcellular Localisation pipeline.
</p>
</body>
</html>"""

html_path = RESULTS_DIR / "interpretation_report.html"
html_path.write_text(html, encoding="utf-8")
print(f"Saved: {html_path}")